# Hospital Patient Flow Demand Analysis¶
### Second Notebook: Operational Data Enrichment and Feature Engineering

This second phase focuses on enriching the hospital dataset with additional operational variables and engineered features to complement the analysis of healthcare operations.

The objective is to transform patient encounter data into a structured analytical framework capable of representing workload patterns, patient flow characteristics, and operational context.

Since the original dataset provides limited information on workforce availability and staffing characteristics, complementary variables will be generated using transparent and documented assumptions. These additional features will enhance the representation of operational dynamics while maintaining reproducibility.

Feature engineering will focus on creating operational indicators that can support different analytical applications, including demand analysis, operational scenarios, and resource allocation studies.

The overall goal is to complement the exploratory analysis by transforming raw patient-level information into a richer operational dataset that can support further studies in healthcare operations.

**Author:** J-F J  
**Date:** July 2026  
**Dataset:** Healthcare Analytics Patient Flow Data (Kaggle)

## 1-Data Loading and Overview

In [1]:
import kagglehub
import pandas as pd

# Download latest version
path = kagglehub.dataset_download("hassanjameelahmed/healthcare-analytics-patient-flow-data")

data_path = "/kaggle/input/datasets/hassanjameelahmed/healthcare-analytics-patient-flow-data/healthcare_analytics_patient_flow_data.csv"

# Clone utility repository
!rm -rf /kaggle/working/jfj-utils
!git clone https://github.com/jfjutras07/jfj-utils.git

# Add to Python path
import sys
sys.path.append("/kaggle/working/jfj-utils")

Cloning into 'jfj-utils'...
remote: Enumerating objects: 3901, done.
remote: Counting objects: 100% (330/330), done.
remote: Compressing objects: 100% (184/184), done.
remote: Total 3901 (delta 272), reused 146 (delta 146), pack-reused 3571 (from 3)
Receiving objects: 100% (3901/3901), 1.30 MiB | 6.25 MiB/s, done.
Resolving deltas: 100% (2565/2565), done.


In [2]:
# Overview
df = pd.read_csv(data_path)
df.head()

,Patient Id,Patient Admission Date,Patient Admission Time,Merged,Patient Gender,Patient Age,Patient Race,Department Referral,Patient Admission Flag,Patient Satisfaction Score,Patient Waittime
0,780-96-6113,9/9/2024,9:25:00 AM,W. Breede,Female,63,African American,NaN,Not Admission,5.0,32
1,714-35-6722,9/9/2024,4:42:00 PM,Y. Baldetti,Male,31,Asian,Orthopedics,Not Admission,NaN,22
2,571-85-3714,9/9/2024,12:14:00 AM,M. Semerad,Male,75,White,General Practice,Not Admission,NaN,16
3,404-43-9499,9/9/2024,8:33:00 PM,K. Blaydes,Male,79,African American,General Practice,Admission,NaN,38
4,552-51-5855,9/9/2024,7:25:00 PM,F. Dickerson,Female,24,African American,NaN,Admission,NaN,36


## 2-Synthetic Operational Assumptions

The original dataset captures patient encounters but does not contain all information required to represent operational characteristics such as workforce structure, staffing availability, and scheduling context. Therefore, complementary datasets and engineered features will be introduced using transparent assumptions.

Together, these components provide a richer representation of hospital operations and allow the exploration of different analytical perspectives related to demand patterns, operational scenarios, and resource planning.

| Component | Purpose | Data Source | Examples |
|---|---|---|---|
| Original Patient Data | Describe individual patient encounters | Original dataset | Age, admission, wait time, referral, arrival time |
| Simulated Operational Data | Represent workforce characteristics and staffing constraints | Simulated assumptions | Professional categories, staffing levels, costs, shift structures |
| Engineered Features | Create decision-relevant variables from observed and simulated data | Original + simulated data | Shift, weekend, age group, hourly demand, workload indicators |
| Forecasting Dataset | Aggregate patient demand for time series analysis | Engineered features | Daily arrivals, hourly arrivals, departmental demand |
| Optimization Dataset | Support prescriptive decision models | All previous components | Demand, workforce availability, costs, decision variables |

**Hospital Workforce**

Healthcare workforce planning requires understanding the availability of different professional categories across departments. This table simulates a simplified staffing structure including physicians, nurses, nursing assistants, and physiotherapists. Real hospital workforce models involve additional roles and constraints; this representation focuses on the main categories required for resource allocation analysis.

In [3]:
# Simulated workforce by department and professional category
df_workforce = pd.DataFrame({
    "Department Referral": [
        "General Practice",
        "Orthopedics",
        "Physiotherapy",
        "Cardiology",
        "Neurology",
        "Gastroenterology",
        "Renal"
    ],
    "General Practitioners": [
        8, 0, 0, 0, 0, 0, 0
    ],
    "Specialist Physicians": [
        0, 8, 6, 7, 6, 5, 4
    ],
    "Nurses": [
        30, 18, 12, 16, 14, 10, 8
    ],
    "Nursing Assistants": [
        15, 10, 8, 8, 7, 6, 5
    ],
    "Physiotherapists": [
        2, 4, 6, 2, 2, 0, 0
    ]
})

df_workforce

,Department Referral,General Practitioners,Specialist Physicians,Nurses,Nursing Assistants,Physiotherapists
0,General Practice,8,0,30,15,2
1,Orthopedics,0,8,18,10,4
2,Physiotherapy,0,6,12,8,6
3,Cardiology,0,7,16,8,2
4,Neurology,0,6,14,7,2
5,Gastroenterology,0,5,10,6,0
6,Renal,0,4,8,5,0


**Workforce Cost Parameters**

Operational decisions require considering not only workforce availability but also the cost associated with each professional category. This table simulates simplified cost parameters that will support future optimization models by enabling cost-based resource allocation scenarios.

In [4]:
# Simulated workforce cost parameters
df_staff_cost = pd.DataFrame({
    "Professional Category": [
        "General Practitioners",
        "Specialist Physicians",
        "Nurses",
        "Nursing Assistants",
        "Physiotherapists"
    ],
    "Hourly Cost": [
        120,
        150,
        50,
        35,
        60
    ]
})

# Preview
df_staff_cost

,Professional Category,Hourly Cost
0,General Practitioners,120
1,Specialist Physicians,150
2,Nurses,50
3,Nursing Assistants,35
4,Physiotherapists,60


**Workforce Schedule**

Workforce planning requires representing the temporal availability of healthcare professionals. This table simulates simplified shift structures to support future resource allocation models.

In [5]:
# Simulated workforce schedule options

df_staff_schedule = pd.DataFrame({
    "Shift Type": [
        "Day Shift",
        "Evening Shift",
        "Night Shift"
    ],
    "Start Time": [
        "07:00",
        "15:00",
        "23:00"
    ],
    "End Time": [
        "15:00",
        "23:00",
        "07:00"
    ],
    "Duration Hours": [
        8,
        8,
        8
    ],
    "Cost Multiplier": [
        1.0,
        1.0,
        1.2
    ]
})

df_staff_schedule

,Shift Type,Start Time,End Time,Duration Hours,Cost Multiplier
0,Day Shift,07:00,15:00,8,1.0
1,Evening Shift,15:00,23:00,8,1.0
2,Night Shift,23:00,07:00,8,1.2


## 3-Operational Feature Engineering

Following the data preparation and exploratory analysis performed in the first notebook, operational features are engineered to transform individual patient encounters into structured analytical variables.

These features capture temporal patterns, patient segmentation, and operational characteristics related to hospital activity. The resulting patient-level dataset will then be aggregated into operational demand indicators to provide additional insight into workload patterns and operational variability.

In [6]:
# Convert date and time variables
df["Patient Admission Date"] = pd.to_datetime(
    df["Patient Admission Date"],
    dayfirst=True,
    errors="coerce"
)

df["Patient Admission Time"] = pd.to_datetime(
    df["Patient Admission Time"],
    format="%I:%M:%S %p",
    errors="coerce"
)

# Temporal operational features
df["Admission Hour"] = (
    df["Patient Admission Time"]
    .dt.hour
)

df["Admission Day"] = (
    df["Patient Admission Date"]
    .dt.day_name()
)

# Preserve datetime format for forecasting compatibility
df["Admission Date"] = (
    df["Patient Admission Date"]
    .dt.normalize()
)

df["Admission Month"] = (
    df["Patient Admission Date"]
    .dt.to_period("M")
    .astype(str)
)

# Weekend indicator
df["Is Weekend"] = (
    df["Patient Admission Date"]
    .dt.dayofweek
    .isin([5, 6])
)

# Shift assignment based on arrival time
def assign_shift(hour):
    if 7 <= hour < 15:
        return "Day Shift"
    elif 15 <= hour < 23:
        return "Evening Shift"
    else:
        return "Night Shift"

df["Shift Type"] = (
    df["Admission Hour"]
    .apply(assign_shift)
)

# Age groups for operational segmentation
df["Age Group"] = pd.cut(
    df["Patient Age"],
    bins=[0, 18, 35, 50, 65, 80],
    labels=[
        "0-18",
        "19-35",
        "36-50",
        "51-65",
        "66+"
    ]
)

# Preview engineered features
df[
    [
        "Patient Admission Date",
        "Patient Admission Time",
        "Admission Hour",
        "Admission Day",
        "Admission Date",
        "Admission Month",
        "Is Weekend",
        "Shift Type",
        "Age Group"
    ]
].head()

,Patient Admission Date,Patient Admission Time,Admission Hour,Admission Day,Admission Date,Admission Month,Is Weekend,Shift Type,Age Group
0,2024-09-09,1900-01-01 09:25:00,9,Monday,2024-09-09,2024-09,False,Day Shift,51-65
1,2024-09-09,1900-01-01 16:42:00,16,Monday,2024-09-09,2024-09,False,Evening Shift,19-35
2,2024-09-09,1900-01-01 00:14:00,0,Monday,2024-09-09,2024-09,False,Night Shift,66+
3,2024-09-09,1900-01-01 20:33:00,20,Monday,2024-09-09,2024-09,False,Evening Shift,66+
4,2024-09-09,1900-01-01 19:25:00,19,Monday,2024-09-09,2024-09,False,Evening Shift,19-35


## 4-Operational Demand Indicators

The engineered patient-level features are aggregated into operational demand indicators to represent hospital workload patterns over time and across clinical dimensions.

These indicators transform individual patient encounters into measurable representations of demand, workload distribution, and operational variability. They provide complementary information that can support further analytical studies in healthcare operations.

**Temporal Demand Aggregation**

In [7]:
# Create hourly patient demand.
# This aggregation captures arrival patterns throughout the day
# and supports analysis of short-term workload variations.
df_hourly_demand = (
    df.groupby("Admission Hour")
    .size()
    .reset_index(name="Patient Volume")
)

# Create daily patient demand.
# Each row represents the total number of patient arrivals observed per day.
# The resulting dataset represents daily patient demand patterns for temporal analysis.
df_daily_demand = (
    df.groupby("Admission Date")
    .size()
    .reset_index(name="Patient Volume")
    .sort_values("Admission Date")
)

# Ensure continuous daily frequency.
# Missing dates correspond to days without recorded arrivals and are filled with zero demand.
# This regular structure is required for forecasting algorithms.
df_daily_demand = (
    df_daily_demand
    .set_index("Admission Date")
    .asfreq("D", fill_value=0)
    .reset_index()
)

**Workforce-Oriented Demand Aggregation**

In [8]:
# Demand by shift.
# Represents workload distribution across workforce scheduling periods.
df_shift_demand = (
    df.groupby("Shift Type")
    .size()
    .reset_index(name="Patient Volume")
)

# Demand by department.
# Identifies workload distribution across clinical services.
df_department_demand = (
    df.groupby("Department Referral")
    .size()
    .reset_index(name="Patient Volume")
)

# Demand by department and shift.
# This represents workload distribution across clinical areas and operational periods.
df_department_shift_demand = (
    df.groupby(
        [
            "Department Referral",
            "Shift Type"
        ]
    )
    .size()
    .reset_index(name="Patient Volume")
)

**Recurring Scheduling Patterns**

In [9]:
# Demand by weekday and shift.
# Captures recurring workload patterns that can support scheduling scenarios.
df_weekday_shift_demand = (
    df.groupby(
        [
            "Admission Day",
            "Shift Type"
        ]
    )
    .size()
    .reset_index(name="Patient Volume")
)

## 5-Operational Dataset Validation

**Demand Consistency Checks**

In [10]:
# Verify that aggregated daily demand matches the original number of patient encounters.
# The total patient volume should remain equal after aggregation.

daily_volume_check = (
    df_daily_demand["Patient Volume"]
    .sum()
)

original_volume = len(df)

print(f"Original patient encounters: {original_volume}")
print(f"Aggregated patient volume: {daily_volume_check}")

Original patient encounters: 9216
Aggregated patient volume: 9216


**Operational Dataset Overview**

In [11]:
# Display the structure of the operational demand datasets.
demand_datasets = {
    "Hourly Demand": df_hourly_demand,
    "Daily Demand": df_daily_demand,
    "Shift Demand": df_shift_demand,
    "Department Demand": df_department_demand,
    "Department-Shift Demand": df_department_shift_demand,
    "Weekday-Shift Demand": df_weekday_shift_demand
}

for name, dataset in demand_datasets.items():
    print(f"\n{name}")
    print(dataset.shape)
    display(dataset.head())


Hourly Demand
(24, 2)


,Admission Hour,Patient Volume
0,0,406
1,1,372
2,2,376
3,3,385
4,4,384



Daily Demand
(579, 2)


,Admission Date,Patient Volume
0,2023-04-01,19
1,2023-04-02,13
2,2023-04-03,14
3,2023-04-04,9
4,2023-04-05,19



Shift Demand
(3, 2)


...

,Department Referral,Patient Volume
0,Cardiology,248
1,Gastroenterology,178
2,General Practice,1840
3,Neurology,193
4,Orthopedics,995



Department-Shift Demand
(21, 3)


,Department Referral,Shift Type,Patient Volume
0,Cardiology,Day Shift,82
1,Cardiology,Evening Shift,89
2,Cardiology,Night Shift,77
3,Gastroenterology,Day Shift,61
4,Gastroenterology,Evening Shift,54



Weekday-Shift Demand
(21, 3)


,Admission Day,Shift Type,Patient Volume
0,Friday,Day Shift,437
1,Friday,Evening Shift,424
2,Friday,Night Shift,449
3,Monday,Day Shift,424
4,Monday,Evening Shift,432


**Department Referral Data Limitation**

Department-level demand indicators rely only on patient encounters with an identified department referral. In the original dataset, 5,400 encounters do not contain a referral destination; however, these records remain valid hospital visits and are included in global demand analyses. Since their clinical destination cannot be determined from the available information, they are excluded from department-specific workload estimation and workforce allocation models. Consequently, workforce optimization will focus on the seven departments with observed referral information.

In [12]:
# Identify encounters with and without a recorded department referral.
# Patients without a referral remain included in global demand analysis
# but are excluded from department-level workforce allocation models.

df["Has Department Referral"] = (
    df["Department Referral"]
    .notna()
)

df["Has Department Referral"].value_counts()

Has Department Referral
False    5400
True     3816
Name: count, dtype: int64

The validation confirms that department-level workload estimation is based on 3,816 encounters with identified referral destinations, while the remaining 5,400 encounters are retained for hospital-wide demand analysis. This distinction ensures that workforce optimization relies only on observable department-level demand information.

## 6-Summary - Notebook 2

| Element | Objective | Methods / Tools | Key Findings | Decision Support Implications |
|---|---|---|---|---|
| Data Loading and Overview | Reuse and structure hospital encounter data | Dataset loading and initial inspection | 9,216 patient encounters successfully loaded with operational and clinical variables | Provides the foundation for operational feature engineering |
| Synthetic Operational Assumptions | Complement the original dataset with operational context | Simulated workforce, cost parameters, and shift structures | Workforce availability, staffing categories, and scheduling characteristics were introduced using documented assumptions | Enriches the operational representation of hospital activities |
| Operational Feature Engineering | Transform patient encounters into operational variables | Temporal features, shift assignment, weekend indicators, age segmentation | Patient-level operational features created to represent arrival patterns and workload characteristics | Supports the analysis of patient flow and operational demand |
| Operational Demand Indicators | Aggregate patient-level data into workload measures | Aggregation by time, shift, department, and scheduling patterns | Demand indicators created at multiple operational levels | Provides structured indicators for demand analysis and operational planning |
| Operational Dataset Validation | Verify consistency and analytical readiness | Volume checks, dataset structure validation, referral availability assessment | Aggregations preserved patient volumes; department-level analysis restricted to 3,816 encounters with identified referrals | Ensures reliable interpretation of operational demand indicators |
| Data Limitations and Modeling Scope | Define analytical boundaries | Missing referral analysis and methodological assumptions | Patients without department referral remain included in global demand analysis but excluded from department-level indicators | Improves transparency and prevents unsupported operational assumptions |
| Overall Conclusion | Summarize operational data enrichment | Evidence-based operational data enrichment | Dataset enriched with operational variables and demand indicators | Provides complementary datasets for subsequent time series analysis, optimization case studies, and decision analysis |

## 7-Data Export

In [13]:
# Export all validated operational datasets into a single Excel workbook.
# Each worksheet represents a specific analytical component used in
# forecasting, simulation, or workforce optimization.

output_file = "hospital_operational_datasets.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    
    df.to_excel(
        writer,
        sheet_name="Patient Features",
        index=False
    )

    df_daily_demand.to_excel(
        writer,
        sheet_name="Daily Demand",
        index=False
    )

    df_hourly_demand.to_excel(
        writer,
        sheet_name="Hourly Demand",
        index=False
    )

    df_department_shift_demand.to_excel(
        writer,
        sheet_name="Department Shift Demand",
        index=False
    )

    df_workforce.to_excel(
        writer,
        sheet_name="Workforce Structure",
        index=False
    )

    df_staff_cost.to_excel(
        writer,
        sheet_name="Workforce Costs",
        index=False
    )

    df_staff_schedule.to_excel(
        writer,
        sheet_name="Workforce Schedule",
        index=False
    )

print("Operational datasets exported successfully.")

Operational datasets exported successfully.
